New notebook to split 4 channel stacks into a single cell stack and the other 3 channels into another file
Also, remove bad z-planes at beginning (or end)
and downsample cell stack, 4-fold. Assume this is channel 1. 

Pseudocode:
import needed stuff

navigate to directory
get cell channel [1]
get first z-plane to keep [1]
get last z-plane to keep [-1]

make output subdirectory (if doesnt exist) for cell stacks
make output subdirectory (if doesnt exist) for RNAnuc stacks
for each file in directory
if its a tiff
    read it in
    assign new RNAnuc stack, keeping appropriate channels and z-planes
    assign new cell stack, keeping appropriate channel and z-planes
    downsample cell stack
    make filenames including subdirectories
    write new RNAnuc file
    write new cell stack
???do I want to also keep the non-downsampled cell stack???


Feb 9: trying to get rid of calls to aicimageio, since I don't think I need to be able to open zen files and it was choking
on files on hard drive. Also there is a dependency incompatability when installing 'pip install "pandas[all]"'

apparently difference between ome tif and tif is only that the former can exceed 4GB, so probably don't need this. I think that means I can read and write tiffs using skimage.

Saving this as splitAndRescale3 -- will use csv file to find z planes, instead of getting from user

In [1]:
import os
import numpy as np
import pandas as pd
from skimage.transform import rescale
from skimage.io import imread, imsave, imshow
from skimage.filters import gaussian
from tkinter import Tk
from tkinter import filedialog
import tifffile as tf
import dask
from skimage import io
import cv2 as cv

dask.config.set({"array.slicing.split_large_chunks": False})
#fTypes = (".tif", ".czi")
fTypes = (".tif")                 # stop allowing czi files, since all converted to tifs allready

# 0.9 was too high-- many pixels of each granule overlap background--trying 0.25

This section is for splitting prior to running cellpose with runCellpose.ipynb
Below is a section for upscaling the label files output by cellpose

In [9]:
root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
dirName = filedialog.askdirectory(initialdir=strtPath)

#print(dirName)
cellFold = dirName + "/cell"
orCellFold = dirName + "/cell/origSize"
nucFold = dirName + "/nuc"
labFold = dirName + "/cell/cLabs"
rnaFold = dirName + "/rna"
if not os.path.exists(cellFold):  
# Create directory for storing the tiff files for cell channels 
   os.mkdir(cellFold)  
if not os.path.exists(orCellFold):  
# Create directory for storing the tiff files for cell channels 
   os.mkdir(orCellFold)  
if not os.path.exists(nucFold):  
# Create directory for storing the tiff files for nuc channels 
   os.mkdir(nucFold)  
if not os.path.exists(labFold):  
# Create directory for storing the tiff files for nuc channels 
   os.mkdir(labFold)  
if not os.path.exists(rnaFold):  
# Create directory for storing the tiff files for nuc channels 
   os.mkdir(rnaFold)  


In [5]:
#csvFile = filedialog.askopenfilename()
csvFile = "/Users/nelsonlab/Documents/forFISHquant/RNAScopeImagelog.csv"
#open pandas file
logTable = pd.read_csv(csvFile,index_col='ImageFile')
#logTable.set_index('ImageFile')
# pandas indexing: df.loc[row_indexer,column_indexer]
# z-start, end for a given filename should be: logTable[filename, "z-start"] or logTable[filename, "z-end"] 

#print (type(strt),type(nd))

In [6]:
# channels for cell, nucleus are hard coded, other chans are rna
c = 1
n = 2
rna1 = 0
rna2 = 3
#notcn = (0, 3)
#fname = '012324_K913_D1_RH_D3_K9.tif'
#logTable.loc['012324_K913_D1_RH_D3_K9.tif', 'z-start']
#logTable.index

In [13]:
# channels for cell, nucleus are hard coded, other chans are rna, C1 only
c = 1
n = 2
rna1 = 0
#notcn = (0, 3)
#fname = '012324_K913_D1_RH_D3_K9.tif'
#logTable.loc['012324_K913_D1_RH_D3_K9.tif', 'z-start']
#logTable.index

In [6]:
subrna1=False #Subtract RNA Channel 2 from 1?
subrna2=False #Subtract RNA Channel 1 from 2?
subrna2_2x=False #Subtract RNA Channel 1 from 2 a second time? (extreme measures)

#print(dirName)
#print (os.listdir(dirName))
for filename in os.listdir(dirName):
   if filename.endswith(".tif") and not filename.startswith("."):
      print (filename)
      base, ext = os.path.splitext(filename)
      if filename in logTable.index:
         zstrt = logTable.loc[filename, 'z-start']
         znd = logTable.loc[filename, 'z-end']
      else:
         zstrt = 1
         znd = -1
         print("NOT IN TABLE, taking all z")
#   print("base=",base," ext=", ext)
      path = os.path.join(dirName, filename)
      orCellFile = "C1_" + filename
      orCellPath = os.path.join(orCellFold, orCellFile)
      img = imread(path)                               # read the original image file
      print("SHAPE= ",img.shape)
      nz = img.shape[0]
      if znd== -1:              # otherwise, leave it as is
         znd=nz
      zstrt = zstrt-1           # correct for 0-based, for znd this correction and fact that slices are open ended (don't include endpoint) cancel out.
      print("retrieving z from ",zstrt," to ",znd-1)
# NOTE: format is Z, Y, X, C: from shape, assume reading and writing does not transpose this--but
# might need to check after writing
#Downscaling cell image for Cellpose       
      cell_chan = img[int(zstrt):int(znd), :, :, c]
      if not os.path.exists(orCellPath):
         imsave(orCellPath, cell_chan, check_contrast=False)
      ds_cell = rescale(cell_chan, (1, 0.25, 0.25), anti_aliasing=True)
      dsG = gaussian(ds_cell, sigma=(0,1.3, 1.3), truncate = 3.5)
      cfile = "DS" + str(c) + "_" + base + ".tif"
      cpath = os.path.join(cellFold,cfile)
      if not os.path.exists(cpath):
         imsave(cpath, dsG, check_contrast=False)
          
#Save nucleus file          
      nfile = "nuc_" + base + ".tif"
      npath = os.path.join(nucFold, nfile)
# NOTE: Have to get number of z slices in data, then use z1, z2 as indices into that
      nuc_chan = img[int(zstrt):int(znd), :, :, n]
      if not os.path.exists(npath):
         imsave(npath, nuc_chan, check_contrast=False)
          
#perform reciprocal subtraction of RNA channels        
      rnafile1 = "rna1_" + base + ".tif"
      rpath1 = os.path.join(rnaFold, rnafile1)  
      rna_chan1 = img[int(zstrt):int(znd), :, :, rna1]  
      
      rnafile2 = "rna2_" + base + ".tif"
      rpath2 = os.path.join(rnaFold, rnafile2)
      rna_chan2 = img[int(zstrt):int(znd), :, :, rna2]
       
      temp1 = rna_chan1.astype(np.int32)
      temp2 = rna_chan2.astype(np.int32)
      if subrna1:
          rna_chan1 = np.clip(temp1 - temp2, 0, 255).astype(np.uint8)
      imsave(rpath1, rna_chan1, check_contrast=False)
      if (subrna2 and not(subrna2_2x)):
          rna_chan2 = np.clip(temp2 - temp1, 0, 255).astype(np.uint8)
      if (subrna2_2x and subrna2):
          temp2 = np.clip(temp2 - temp1, 0, 255)
          rna_chan2 = np.clip(temp2 - temp1, 0, 255).astype(np.uint8)
      imsave(rpath2, rna_chan2, check_contrast=False)

040924_ScrGFP_B1_RH_D3_K9.tif
NOT IN TABLE, taking all z
SHAPE=  (21, 3008, 3008, 3)
retrieving z from  0  to  20


NameError: name 'rna1' is not defined

In [15]:
#C1 Only

#print(dirName)
#print (os.listdir(dirName))
for filename in os.listdir(dirName):
   if filename.endswith(".tif") and not filename.startswith("."):
      print (filename)
      base, ext = os.path.splitext(filename)
      if filename in logTable.index:
         zstrt = logTable.loc[filename, 'z-start']
         znd = logTable.loc[filename, 'z-end']
      else:
         zstrt = 1
         znd = -1
         print("NOT IN TABLE, taking all z")
#   print("base=",base," ext=", ext)
      path = os.path.join(dirName, filename)
      orCellFile = "C1_" + filename
      orCellPath = os.path.join(orCellFold, orCellFile)
      img = imread(path)                               # read the original image file
      print("SHAPE= ",img.shape)
      nz = img.shape[0]
      if znd== -1:              # otherwise, leave it as is
         znd=nz
      zstrt = zstrt-1           # correct for 0-based, for znd this correction and fact that slices are open ended (don't include endpoint) cancel out.
      print("retrieving z from ",zstrt," to ",znd-1)
# NOTE: format is Z, Y, X, C: from shape, assume reading and writing does not transpose this--but
# might need to check after writing
#Downscaling cell image for Cellpose       
      cell_chan = img[int(zstrt):int(znd), :, :, c]
      if not os.path.exists(orCellPath):
         imsave(orCellPath, cell_chan, check_contrast=False)
      ds_cell = rescale(cell_chan, (1, 0.25, 0.25), anti_aliasing=True)
      dsG = gaussian(ds_cell, sigma=(0,1.3, 1.3), truncate = 3.5)
      cfile = "DS" + str(c) + "_" + base + ".tif"
      cpath = os.path.join(cellFold,cfile)
      if not os.path.exists(cpath):
         imsave(cpath, dsG, check_contrast=False)
          
#Save nucleus file          
      nfile = "nuc_" + base + ".tif"
      npath = os.path.join(nucFold, nfile)
# NOTE: Have to get number of z slices in data, then use z1, z2 as indices into that
      nuc_chan = img[int(zstrt):int(znd), :, :, n]
      if not os.path.exists(npath):
         imsave(npath, nuc_chan, check_contrast=False)
               
#Save RNA file 
       
      rnafile1 = "rna1_" + base + ".tif"
      rpath1 = os.path.join(rnaFold, rnafile1)
      rna_chan1 = img[int(zstrt):int(znd), :, :, rna1]
      imsave(rpath1, rna_chan1, check_contrast=False)
       


040924_ScrGFP_B1_RH_D3_K9.tif
NOT IN TABLE, taking all z
SHAPE=  (21, 3008, 3008, 3)
retrieving z from  0  to  20
040924_ScrGFP_B2_RH_D3_K9.tif
NOT IN TABLE, taking all z
SHAPE=  (28, 3008, 3008, 3)
retrieving z from  0  to  27
040924_ScrGFP_B3_LH_D3_K9.tif
NOT IN TABLE, taking all z
SHAPE=  (18, 3008, 3008, 3)
retrieving z from  0  to  17
040924_ScrGFP_B4_LH_D3_K9.tif
NOT IN TABLE, taking all z
SHAPE=  (21, 3008, 3008, 3)
retrieving z from  0  to  20


Removing cellpose, but will add upscaling the cellular masks here. Then will need to count overlaps.

In [12]:
from skimage.segmentation import clear_border
from napari_simpleitk_image_processing import label_statistics
from skimage.measure import regionprops_table

Icutoff = 10 #remove bottom x% of signal (default: 5)
Bcutoff = 5 #remove cells where y% is on border (default: 2)

root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
dirName = filedialog.askdirectory(initialdir=strtPath)

#print(imgFile, "type=", type(imgFile))
cellFold = dirName + "/cell"
labFold = os.path.join(cellFold,"cLabs")
orCellFold = os.path.join(cellFold,"origSize")
upPath = os.path.join(labFold,"upscaled")

if not os.path.exists(upPath):  
# Create directory for storing the tiff files for cell channels 
   os.mkdir(upPath)

#Main loop - load in label/img files, filter out bad calls, save cell (img) properties, upscale label file
for filename in os.listdir(cellFold):
   if filename.endswith(".tif"):
       print(filename)
       filePath = os.path.join(cellFold, filename)
       labName = filename.replace(".tif","_cp_masks.tif")
       labPath = os.path.join(labFold, labName)
       if os.path.exists(labPath):
           theimg = io.imread(filePath)
           lab = io.imread(labPath)
           labfilt = lab.copy()
           lab_pad = np.pad(
               lab,
               ((1, 1), (0, 0), (0, 0)),
               mode='constant',
               constant_values=0,
           )
           img_pad = np.pad(
               theimg,
               ((1, 1), (0, 0), (0, 0)),
               mode='constant',
               constant_values=0,
           )
           regionprops = label_statistics(img_pad, lab_pad, True, True, True, True)
       #Find and filter false positives (bottom10% of summed intensity) 
           cutoff = ((regionprops["sum"].max() - regionprops["sum"].min())/100) * Icutoff
           cellidx = regionprops[regionprops["sum"] < cutoff]["label"].to_numpy()
       #Find and filter border cells (5% overlap with borders)
           cellidx2 = regionprops[(regionprops["number_of_pixels_on_border"]/regionprops["number_of_pixels"]) > (Bcutoff/100)]["label"].to_numpy()
           cellidx = np.union1d(cellidx, cellidx2)

           labfilt[np.where(np.isin(labfilt, cellidx))] = 0

           #regionprops = regionprops[~regionprops["label"].isin(cellidx)]
           regionprops.to_csv(os.path.splitext(filePath)[0] + "_cellprops.csv")
           print("saved label statistics")
           
           if labName.endswith("_cp_masks.tif") and labName.startswith("DS"):
               lab_up = rescale(labfilt, (1, 4, 4), anti_aliasing=False, preserve_range=True, order=0)
               c4fname = "C" + labName[2:]
               c4fpath = os.path.join(upPath, c4fname)
               imsave(c4fpath, lab_up, check_contrast=False)
               print("saved rescaled files")
               
               orCellfile = c4fname.replace("_cp_masks.tif",".tif")
               orCellpath = os.path.join(orCellFold,orCellfile)
               orCell = imread(orCellpath)
               cLabProps = regionprops_table(lab_up, orCell, properties=("label","centroid", "area", "intensity_max", "intensity_mean", "num_pixels"))           
               dfCellProps = pd.DataFrame(cLabProps)
               dfCellProps.to_csv(os.path.splitext(orCellpath)[0] + "_cellprops.csv")
               print("saved filtered label statistics")
           else:
               print ("wrong filetype--not DS cp_mask", filename)
               continue
       else:
           print("label file not found")
           continue



DS1_040924_ScrGFP_B1_RH_D3_K9.tif
saved label statistics
saved rescaled files
saved filtered label statistics
DS1_040924_ScrGFP_B2_RH_D3_K9.tif
saved label statistics
saved rescaled files
saved filtered label statistics
DS1_040924_ScrGFP_B3_LH_D3_K9.tif
saved label statistics
saved rescaled files
saved filtered label statistics
DS1_040924_ScrGFP_B4_LH_D3_K9.tif
saved label statistics
saved rescaled files
saved filtered label statistics


In [35]:
#For viewing individual image/label pairs (select image file to load label)
import napari

root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
imgFile = filedialog.askopenfilename(initialdir=strtPath)

imgname = os.path.basename(imgFile)
orCellname = imgname.replace(".tif","_cp_masks.tif")
if imgname.startswith("DS1"):
    imgPath = os.path.dirname(imgFile)
    labFile = os.path.join(imgPath,"cLabs",orCellname)
else:
    imgPath = os.path.dirname(os.path.dirname(imgFile))
    labFile = os.path.join(imgPath,"cLabs/upscaled",orCellname)
    
if os.path.exists(labFile):
    theimg = io.imread(imgFile)
    lab = io.imread(labFile)
    viewer =  napari.view_image(theimg, name="theimg")
    labels_layer = viewer.add_labels(lab, name='segmentation')
else: print("Label file not found")

2024-04-18 15:54:18.703 python[78309:19370711] +[CATransaction synchronize] called within transaction
2024-04-18 15:54:18.735 python[78309:19370711] +[CATransaction synchronize] called within transaction
2024-04-18 15:54:36.485 python[78309:19370711] +[CATransaction synchronize] called within transaction


KeyboardInterrupt: 